<a href="https://colab.research.google.com/github/sonyxoxo/ipd/blob/main/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5_%D1%84%D0%B8%D0%B7%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D1%85_%D0%B2%D0%B5%D0%BB%D0%B8%D1%87%D0%B8%D0%BD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Этап 3. Преобразование физических величин

Важно не путать **мощность** с **энергией**. Для средней мощности `P` за интервал в одну минуту энергия `E = P × (1/60 ч)`.

## Имена переменных и единицы измерения

- `source` — файл измерений в репозитории; `measurements` — его таблица в `pandas`.
- `minutes` — минутная сетка с явно отмеченными пропусками.
- `energy_table` — независимая рабочая копия с расчётными столбцами энергии.
- `Global_active_power` — средняя активная мощность за минуту, **кВт**.
- `active_energy_kwh` — энергия за конкретную минуту, **кВт·ч**, равна `Global_active_power / 60`.
- `Sub_metering_1`/`2`/`3` — энергия, зарегистрированная дополнительными счётчиками за минуту, **Вт·ч**.
- `sub_1_energy_kwh`, `sub_2_energy_kwh`, `sub_3_energy_kwh` — те же показания, переведённые в **кВт·ч** делением на 1000.
- `energy_columns` — список столбцов для сохранения: время и вычисленная энергия.

В UCI для `Global_reactive_power` подписано «киловатт», но в электротехнике единицей реактивной мощности является **квар**. Мы не используем этот столбец для вычисления активной энергии.

Неизвестные измерения остаются `NaN`. Сумма известных минутных энергий неполного месяца **не является** энергией всего месяца.

In [1]:
# Одна настройка пути к репозиторию. Исходные файлы не загружаются вручную.
from pathlib import Path
import subprocess
import sys

repo_dir = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "ipd_analysis.py").is_file()), None)
if repo_dir is None:
    repo_dir = Path("/content/ipd")
    if not (repo_dir / "ipd_analysis.py").is_file():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/sonyxoxo/ipd.git", str(repo_dir)], check=True)
sys.path.insert(0, str(repo_dir))

from ipd_analysis import choose_source, load_measurements, regular_minutes, result_folder, save_csv, save_run_metadata

# Если в data/raw несколько файлов, задайте относительный путь явно:
# source = choose_source("data/raw/household_power_consumption.zip")
source = choose_source()
measurements = load_measurements(source)  # Один pd.read_csv на весь ноутбук.
output_dir = result_folder(source)
output_dir.mkdir(parents=True, exist_ok=True)
print("Источник:", source)
print("Строк:", len(measurements))

Источник: /content/ipd/data/raw/household_power_consumption.zip
Строк: 2075259


## Расчёт и независимая проверка

`convert_energy()` проверяет обратное действие: умножение рассчитанной энергии за одну минуту на 60 восстанавливает исходную среднюю мощность. Данные не заполняются нулями. В этой операции для каждой известной минуты предполагается одинаковый минутный интервал.

In [2]:
from ipd_analysis import convert_energy

minutes = regular_minutes(measurements)
energy_table = convert_energy(minutes)
energy_columns = [name for name in energy_table.columns if name == "timestamp" or name.endswith("_kwh")]
display(energy_table[energy_columns].head(10))
save_csv(energy_table[energy_columns], output_dir / "03_energy_kwh.csv")
print("Рассчитанных минут:", len(energy_table))

,timestamp,active_energy_kwh,sub_1_energy_kwh,sub_2_energy_kwh,sub_3_energy_kwh
0,2006-12-16 17:24:00,0.070267,0.0,0.001,0.017
1,2006-12-16 17:25:00,0.089333,0.0,0.001,0.016
2,2006-12-16 17:26:00,0.089567,0.0,0.002,0.017
3,2006-12-16 17:27:00,0.089800,0.0,0.001,0.017
4,2006-12-16 17:28:00,0.061100,0.0,0.001,0.017
5,2006-12-16 17:29:00,0.058667,0.0,0.002,0.017
6,2006-12-16 17:30:00,0.061700,0.0,0.001,0.017
7,2006-12-16 17:31:00,0.061667,0.0,0.001,0.017
8,2006-12-16 17:32:00,0.061133,0.0,0.001,0.017
9,2006-12-16 17:33:00,0.061033,0.0,0.002,0.016


Рассчитанных минут: 2075259


## Воспроизводимость результата

Последняя ячейка сохраняет `metadata.json` с SHA-256 исходного файла, хешами исполняемого кода, версиями библиотек и датой запуска. Поэтому любой результат можно связать с конкретным источником и версией обработки.

In [3]:
save_run_metadata(source, output_dir, {"notebook": 'преобразование физических величин.ipynb'})

PosixPath('/content/ipd/результаты/household_power_consumption-b916e64b/metadata.json')